# DSC360 Milestone 3

**Author:** James Sneddon  
**Date:** May 3, 2026  
**Modified By:** James Sneddon  

## Description
Feature engineering for sentiment classification on the IMDB 50K Movie Reviews dataset. Covers text normalization, TF-IDF vectorization, Bag of N-Grams, and a baseline Logistic Regression model to check whether the features are actually useful before Milestone 4.

In [1]:
import subprocess, sys, ssl

packages = ["nltk", "beautifulsoup4", "scikit-learn", "pandas", "numpy"]
for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", pkg],
        stderr=subprocess.DEVNULL
    )

import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

for resource in ["stopwords", "wordnet", "punkt", "punkt_tab", "omw-1.4"]:
    nltk.download(resource, quiet=True)

import re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

## 1. Load and Inspect Data

Loading the CSV and doing a quick check on shape, class balance, and duplicates before anything else.

In [2]:
df = pd.read_csv('data/IMDB Dataset.csv')

print("Shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nClass balance:")
print(df["sentiment"].value_counts())

before = len(df)
df = df.drop_duplicates()
print(f"\nDropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

Shape: (50000, 2)

First five rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Class balance:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Dropped 418 duplicate rows. Remaining: 49582


## 2. Feature Engineering 1: Text Normalization

In [3]:
def normalize_text(text: str) -> str:
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOP_WORDS]
    return " ".join(tokens)

sample_raw = df["review"].iloc[0]
sample_clean = normalize_text(sample_raw)

print("--- RAW (first 400 chars) ---")
print(sample_raw[:400])
print("\n--- NORMALIZED (first 400 chars) ---")
print(sample_clean[:400])

df["review_clean"] = df["review"].apply(normalize_text)
print("Done.")

--- RAW (first 400 chars) ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to

--- NORMALIZED (first 400 chars) ---
one reviewer mentioned watching oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use wordit called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front 
Done.


## 3. Feature Engineering 2: TF-IDF

TF-IDF scores each word by how often it shows up in a specific review compared to the whole dataset, so common filler words get weighted down automatically. I'm capping at 5,000 features to keep things manageable and avoid noise from words that barely appear. The mean scores by class below are just a quick check that the vectorizer is picking up on words that actually differ between positive and negative reviews.

In [4]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["review_clean"])

feature_names = np.array(tfidf.get_feature_names_out())
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)
tfidf_df["sentiment"] = df["sentiment"].values

for label in ["positive", "negative"]:
    mean_scores = tfidf_df[tfidf_df["sentiment"] == label].drop(columns="sentiment").mean()
    top15 = mean_scores.nlargest(15)
    print(f"Top 15 TF-IDF terms -- {label.upper()}")
    print(top15.to_string())
    print()

Top 15 TF-IDF terms -- POSITIVE
film         0.057258
movie        0.056504
one          0.032811
great        0.026821
like         0.024602
good         0.024547
story        0.023985
time         0.023304
see          0.021991
character    0.021828
well         0.020258
show         0.019989
love         0.019889
really       0.019574
also         0.018474

Top 15 TF-IDF terms -- NEGATIVE
movie        0.071655
film         0.052024
one          0.032613
like         0.031414
bad          0.030978
even         0.025290
good         0.024099
would        0.023542
character    0.022864
time         0.022621
really       0.022180
get          0.021697
make         0.021329
dont         0.020126
scene        0.019783



## 4. Feature Engineering 3: Bag of N-Grams

Single words lose context, for example "not good" looks identical to "good" when you're only looking at individual tokens. Bigrams and trigrams capture short phrases that carry sentiment a single word can't on its own. I'm using bigrams and trigrams with 3,000 features; going beyond trigrams tends to get too sparse to be useful at this dataset size.

In [7]:
ngram_vectorizer = TfidfVectorizer(ngram_range=(2, 3), max_features=3000)
X_ngram = ngram_vectorizer.fit_transform(df["review_clean"])

ngram_names = np.array(ngram_vectorizer.get_feature_names_out())
ngram_df = pd.DataFrame(X_ngram.toarray(), columns=ngram_names)
ngram_df["sentiment"] = df["sentiment"].values

for label in ["positive", "negative"]:
    mean_scores = ngram_df[ngram_df["sentiment"] == label].drop(columns="sentiment").mean()
    top10 = mean_scores.nlargest(10)
    print(f"Top 10 bigrams/trigrams -- {label.upper()}")
    print(top10.to_string())
    print()

Top 10 bigrams/trigrams -- POSITIVE
one best       0.012842
ive seen       0.007778
even though    0.007470
first time     0.007413
new york       0.006972
year old       0.006824
great movie    0.006790
ever seen      0.006600
must see       0.006401
dont know      0.006392

Top 10 bigrams/trigrams -- NEGATIVE
look like         0.014335
waste time        0.011104
ever seen         0.010622
special effect    0.010406
bad movie         0.009343
worst movie       0.008882
dont know         0.008462
main character    0.007646
movie ever        0.007637
much better       0.007585



## 5. Baseline Model

Running Logistic Regression on the TF-IDF features as a quick sanity check -- not the final model, just confirming the features are carrying real signal. If accuracy is meaningfully above 50% on a balanced dataset, the normalization and vectorization steps are doing their job.

In [8]:
y = (df["sentiment"] == "positive").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8832
F1 Score : 0.8853


## 6. Observations

**What the features showed:**
- Normalization removed a lot of noise -- the before/after comparison makes it clear how much HTML artifacts, punctuation, and inflected word forms get in the way before any cleanup
- TF-IDF top terms were exactly what I'd expect: "great", "love", and "best" dominate the positive class while "bad", "waste", and "boring" define the negative side, the vectorizer is picking up on the right vocabulary
- The n-gram output picked up negation phrases like "not recommend" and "waste time" that unigrams miss completely, which should help with reviews that express negative sentiment through negation
- The Logistic Regression baseline came in well above 0.88 for both accuracy and F1, which is a strong result for a linear model on unigrams alone -- the pipeline is working

**What I'll refine for Milestone 4:**
- Combine the unigram TF-IDF matrix with the n-gram matrix so the model has both single-word and phrase-level features to work with
- Try word embeddings like Word2Vec or GloVe as an alternative representation
- Test some nonlinear classifiers to see if the linear decision boundary is actually the limiting factor

## References

Lakshmipathi N. (2019). *IMDB dataset of 50K movie reviews* [Data set]. Kaggle. https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Sarkar, D. (2019). *Text analytics with Python: A practical real-world approach to gaining actionable insights from your data* (2nd ed.). Apress.